<a href="https://colab.research.google.com/github/rizkyhaksono/llm-vs-slm-lab/blob/main/01-konsep-llm-vs-slm/01_apa_itu_language_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01.01 — Apa Itu Language Model?

**Tujuan**: bangun intuisi level tinggi tentang apa yang sebenarnya dilakukan language model. Setelah notebook ini, kamu bisa jawab pertanyaan "jadi LLM itu sebenarnya ngapain?" dalam 1 paragraf.

**Prasyarat**: notebook 00.01 lulus.

**Catatan**: ini bukan textbook — internals (transformer, attention) di luar scope. Lihat repo [`llm-internals`](../../llm-internals/) kalau mau dalam.

---

## Definisi 1 baris

> **Language model**: program yang, diberi suatu rangkaian kata, memprediksi distribusi probabilitas kata berikutnya.

Itu doang. Yang berubah dari tahun ke tahun:
1. **Dari mana** model belajar distribusi itu (corpus kecil → seluruh internet).
2. **Bagaimana** model menyimpan dan menghitung distribusi itu (count → neural network).
3. **Seberapa besar** parameter yang dipakai untuk capture pola (juta → triliun).

## 0. Bootstrap (jalankan pertama)

Cell standar yang bikin notebook jalan di lokal & Colab. Sama seperti di 00.01.

In [1]:
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_NAME = "llm-vs-slm-lab"
    REPO_URL = "https://github.com/rizkyhaksono/llm-vs-slm-lab.git"  # <-- ganti setelah push
    if not Path(REPO_NAME).exists():
        !git clone {REPO_URL}
    %cd {REPO_NAME}
    !pip install -q torch --index-url https://download.pytorch.org/whl/cpu
    !pip install -q -r requirements.txt

repo_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "requirements.txt").exists():
        repo_root = candidate
        break
assert repo_root is not None, "Tidak ketemu root repo (requirements.txt). Jalankan dari dalam repo."
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print(f"IN_COLAB={IN_COLAB}, repo_root={repo_root}")

Cloning into 'llm-vs-slm-lab'...
remote: Enumerating objects: 51, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 51 (delta 8), reused 51 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (51/51), 40.71 KiB | 926.00 KiB/s, done.
Resolving deltas: 100% (8/8), done.
/content/llm-vs-slm-lab
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 MB 8.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.5/12.5 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.8/118.8 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 9.6 

## 1. Versi paling sederhana: bigram counting

Sebelum ada deep learning, language model dibangun dengan **counting**. Idenya brutal sederhana: kalau di corpus kata "saya" 100 kali diikuti "makan" dan 50 kali diikuti "pulang", maka:

$$P(\text{makan} \mid \text{saya}) = \frac{100}{100 + 50 + \dots} \approx 0.4$$

Mari kita coba di toy corpus.

In [2]:
from collections import Counter, defaultdict

# toy corpus: 5 kalimat sederhana Bahasa Indonesia
corpus = [
    "saya makan nasi",
    "saya makan roti",
    "saya minum kopi",
    "kamu makan nasi",
    "kamu minum teh",
]

# build bigram counts
bigram_counts: dict[str, Counter] = defaultdict(Counter)
for sentence in corpus:
    tokens = sentence.split()
    for prev_tok, next_tok in zip(tokens, tokens[1:]):
        bigram_counts[prev_tok][next_tok] += 1

# print probabilities
print("P(next | prev) di toy corpus:\n")
for prev, nexts in bigram_counts.items():
    total = sum(nexts.values())
    for next_tok, count in nexts.most_common():
        prob = count / total
        print(f"  P({next_tok!r:<8} | {prev!r:<8}) = {prob:.2f}  ({count}/{total})")
    print()

P(next | prev) di toy corpus:

  P('makan'  | 'saya'  ) = 0.67  (2/3)
  P('minum'  | 'saya'  ) = 0.33  (1/3)

  P('nasi'   | 'makan' ) = 0.67  (2/3)
  P('roti'   | 'makan' ) = 0.33  (1/3)

  P('kopi'   | 'minum' ) = 0.50  (1/2)
  P('teh'    | 'minum' ) = 0.50  (1/2)

  P('makan'  | 'kamu'  ) = 0.50  (1/2)
  P('minum'  | 'kamu'  ) = 0.50  (1/2)



Sudah bisa generate teks. Pilih kata awal, lihat distribusi, sampling, ulang.

In [3]:
import random

random.seed(42)

def sample_next(prev: str) -> str | None:
    """Sample next token dari bigram distribution. Return None kalau prev tidak dikenal."""
    if prev not in bigram_counts:
        return None
    nexts = bigram_counts[prev]
    tokens = list(nexts.keys())
    weights = list(nexts.values())
    return random.choices(tokens, weights=weights, k=1)[0]

for _ in range(5):
    seq = ["saya"]
    for _ in range(5):
        nxt = sample_next(seq[-1])
        if nxt is None:
            break
        seq.append(nxt)
    print(" ".join(seq))

saya makan nasi
saya makan nasi
saya minum teh
saya minum kopi
saya makan nasi


**Itu** sudah language model.

Kelemahannya jelas:
- Cuma lihat 1 kata sebelumnya. Tidak ada konteks panjang.
- Kombinasi yang tidak ada di corpus → probability 0 (zero problem).
- Tidak bisa generalisasi: "saya makan apel" tidak akan pernah muncul kalau corpus tidak punya "apel".

## 2. Naikkan: n-gram dengan konteks lebih panjang

Bukan cuma 1 kata sebelumnya, tapi 2, 3, 4 kata. Ini namanya trigram, 4-gram, dst. Model semakin akurat, tapi:

- Storage meledak: untuk 5-gram di corpus 1 juta kata, kombinasi unik bisa ratusan juta.
- Sparsity makin parah: makin panjang context, makin sering kombinasi belum pernah dilihat.

Ini batas teknik counting. Tahun 2010-an, counting-based n-gram model **sudah dikalahkan** oleh pendekatan baru: **neural language model**.

## 3. Loncatan besar: neural language model

Idenya berubah total:

1. Setiap token diubah jadi **vektor** (embedding) — bukan ID diskrit.
2. Vektor token konteks di-feed ke **neural network**.
3. Network output **distribusi probability** atas semua kemungkinan token berikut.
4. Train network dengan banyak teks supaya output nya cocok dengan distribusi sebenarnya.

Keuntungan:
- Kata dengan makna mirip → vektor mirip → behavior mirip. Generalisasi.
- Konteks panjang dihandle oleh arsitektur (attention, LSTM, dst).
- Kapasitas model bisa di-scale dengan menambah parameter.

Arsitektur dominan sekarang: **Transformer** (Vaswani et al. 2017). Detail di [`llm-internals/03`](../../llm-internals/) dan [`llm-internals/04`](../../llm-internals/).

## 4. Mari lihat distribusi probability dari LM beneran

Kita ambil SmolLM2-135M, beri prompt, dan lihat **5 token paling probable** untuk posisi berikutnya.

Ini bagian yang sebagian besar tutorial skip — padahal ini esensi dari language model: distribusi atas vocabulary.

In [4]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "HuggingFaceTB/SmolLM2-135M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32)
model.eval()

print("Model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/861 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Model loaded.


In [5]:
def show_top_k_next_tokens(prompt: str, k: int = 5) -> None:
    """Print k token paling probable untuk posisi setelah prompt."""
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids
    with torch.no_grad():
        outputs = model(input_ids)
    # logits shape: [batch, seq_len, vocab_size]; ambil token terakhir
    last_logits = outputs.logits[0, -1, :]
    probs = F.softmax(last_logits, dim=-1)
    top_p, top_idx = torch.topk(probs, k)

    print(f"Prompt: {prompt!r}")
    print("Top kandidat token berikutnya:")
    for prob, idx in zip(top_p.tolist(), top_idx.tolist()):
        token_str = tokenizer.decode([idx])
        print(f"  P({token_str!r:<20}) = {prob:.4f}")
    print()

show_top_k_next_tokens("Ibu kota Indonesia adalah")
show_top_k_next_tokens("Saya suka makan")
show_top_k_next_tokens("2 + 2 =")

Prompt: 'Ibu kota Indonesia adalah'
Top kandidat token berikutnya:
  P(' k'                ) = 0.0411
  P(' '                 ) = 0.0319
  P(' p'                ) = 0.0295
  P(' men'              ) = 0.0256
  P(' mem'              ) = 0.0221

Prompt: 'Saya suka makan'
Top kandidat token berikutnya:
  P(','                 ) = 0.0725
  P(' k'                ) = 0.0512
  P(' b'                ) = 0.0367
  P(' ke'               ) = 0.0349
  P(' p'                ) = 0.0310

Prompt: '2 + 2 ='
Top kandidat token berikutnya:
  P(' '                 ) = 0.8955
  P(' -'                ) = 0.0213
  P(' ('                ) = 0.0095
  P(' a'                ) = 0.0040
  P(' �'                ) = 0.0039



Perhatikan:
- Untuk "Ibu kota Indonesia adalah", model harusnya kasih probabilitas tinggi ke " Jakarta". (Note: SmolLM2 dilatih dominan corpus English, mungkin agak miss untuk Bahasa.)
- Untuk "2 + 2 =", harusnya " 4" dominan.

Inilah yang dilakukan LM: **distribusi probability atas vocabulary**. Generate teks = ambil sample dari distribusi → append → ulang.

## 5. Kunci: skala data + parameter

Kenapa GPT-4 jauh lebih hebat dari bigram di section 1, padahal **konsep dasarnya sama** (next token prediction)?

Tiga hal yang berubah:

| Aspek | Bigram (kita di section 1) | LLM modern |
|---|---|---|
| Training data | 5 kalimat (~15 kata) | trilyun token (~10^13) |
| Parameter | tabel ~10 entries | 8B–500B+ |
| Konteks | 1 kata sebelumnya | 8k–1M token |
| Compute training | <1 detik | jutaan dollar GPU |

Jadi **emergent capabilities** (math reasoning, coding, multi-bahasa) muncul karena **scale**, bukan karena fundamentalnya beda.

## Refleksi & insight

Yang harus stick di kepala kamu setelah notebook ini:

1. **Language model = next-token predictor.** Itu doang.
2. **Generate teks** = sampling iteratif dari distribusi probability.
3. **Beda LLM dengan bigram di section 1** lebih banyak quantitative (skala) daripada qualitative (konsep).
4. **"Reasoning" yang kelihatan di LLM modern** adalah konsekuensi dari training di trillion token — bukan fitur khusus yang di-program.
5. **Bahasa Indonesia di model multi-bahasa** sering kurang akurat karena corpus dominan English. Akan kita demonstrasikan di notebook 02.

## Latihan mandiri (opsional)

1. **Bandingkan top-5 token** untuk prompt yang sama (mis. "Ibu kota Indonesia adalah") di SmolLM2-135M vs Llama-3.1-8B (via Groq, lihat field `logprobs` kalau di-enable). Apa yang beda? Hint: model lebih besar biasanya lebih confident di prediksi yang benar (probability distribution lebih sharp).
2. **Coba bigram model di corpus yang lebih besar.** Download 1 file novel public domain (mis. Pride and Prejudice dari Project Gutenberg). Apakah generate output nya jadi "masuk akal" dengan corpus lebih besar?

## Lanjut

Sekarang kamu paham apa itu LM. Notebook berikut definisikan formalnya **LLM** vs **SLM** dan kapan dipilih: [02_llm_vs_slm_definisi.ipynb](02_llm_vs_slm_definisi.ipynb).